In [1]:
# Install if needed
# !pip install transformers torch

from transformers import pipeline



d:\dream_machine\AI-content-moderator\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------------
# 1. Load Models (Singletons)
# -------------------------------

# Toxicity model (fast, specific)
toxicity_model = pipeline(
    "text-classification",
    model="unitary/toxic-bert"
)

# Zero-shot model (flexible for multiple labels)
zero_shot_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7815.91it/s]
BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
d:\dream_machine\AI-content-moderator\env\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shubh\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#

In [3]:
# -------------------------------
# 2. Detection Functions
# -------------------------------

def detect_toxicity(text: str) -> float:
    result = toxicity_model(text)[0]

    label = result["label"].lower()
    score = result["score"]

    if "toxic" in label:
        return score
    else:
        return 1 - score


def detect_zero_shot(text: str, candidate_labels: list) -> dict:
    result = zero_shot_model(text, candidate_labels)

    return dict(zip(result["labels"], result["scores"]))


def detect_sexual(text: str) -> float:
    labels = ["sexual content", "normal conversation"]
    result = detect_zero_shot(text, labels)
    return result.get("sexual content", 0.0)


def detect_hate(text: str) -> float:
    labels = ["hate speech", "normal conversation"]
    result = detect_zero_shot(text, labels)
    return result.get("hate speech", 0.0)


def detect_self_harm(text: str) -> float:
    labels = ["self harm", "safe"]
    result = detect_zero_shot(text, labels)
    return result.get("self harm", 0.0)




In [4]:
# -------------------------------
# 3. Aggregator
# -------------------------------

def moderate_text(text: str) -> dict:
    return {
        "toxicity": detect_toxicity(text),
        "sexual": detect_sexual(text),
        "hate": detect_hate(text),
        "self_harm": detect_self_harm(text)
    }




In [5]:
# -------------------------------
# 4. Decision Engine
# -------------------------------

THRESHOLDS = {
    "toxicity": 0.8,
    "sexual": 0.7,
    "hate": 0.7,
    "self_harm": 0.85
}


def decide(scores: dict):
    for label, threshold in THRESHOLDS.items():
        if scores.get(label, 0) > threshold:
            return {
                "status": "BLOCKED",
                "reason": label
            }

    if scores.get("toxicity", 0) > 0.5:
        return {
            "status": "FLAGGED",
            "reason": "toxicity"
        }

    return {
        "status": "APPROVED",
        "reason": None
    }




In [6]:
# -------------------------------
# 5. Moderation Service
# -------------------------------

def moderate(text: str):
    scores = moderate_text(text)
    decision = decide(scores)

    return {
        "text": text,
        "scores": scores,
        "status": decision["status"],
        "reason": decision["reason"]
    }




In [ ]:
# -------------------------------
# 6. Test Cases
# -------------------------------

texts = [
    # "Hello, how are you?",
    # "I hate you",
    # "You are stupid",
    # "Send me nudes",
    # "I want to kill myself",
    # "Let's build something amazing",
    # "I will kill you",
]

for text in texts:
    result = moderate(text)
    print(result)
    print("-" * 60)

{'text': 'nude', 'scores': {'toxicity': 0.5203258395195007, 'sexual': 0.9763498902320862, 'hate': 0.5254191756248474, 'self_harm': 0.18378180265426636}, 'status': 'BLOCKED', 'reason': 'sexual'}
------------------------------------------------------------
